In [1]:
!pip install fastapi uvicorn sqlalchemy google-genai nest-asyncio python-dotenv passlib pyjwt tensorflow pillow pydantic

In [2]:
%%writefile .env
JWT_SECRET_KEY=fallback-secret
JWT_ALGORITHM=HS256
API_KEY=my_secret_skincare_api_key_12345
GEMINI_API_KEY=AIzaSy...your_actual_key_here...

Overwriting .env


In [3]:
%%writefile main.py
import os
import io
import jwt
import numpy as np
from datetime import datetime, timedelta
from PIL import Image

from fastapi import FastAPI, HTTPException, Depends, Security, File, UploadFile, status
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials, APIKeyHeader
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, EmailStr
from passlib.context import CryptContext
from dotenv import load_dotenv

import tensorflow as tf
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session

from google import genai

load_dotenv()

# Environment Variables
JWT_SECRET_KEY = os.getenv("JWT_SECRET_KEY", "fallback-secret")
JWT_ALGORITHM = os.getenv("JWT_ALGORITHM", "HS256")
API_KEY = os.getenv("API_KEY", "my_secret_skincare_api_key_12345")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# Initialize Gemini Client if Key exists
gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

# Database Setup (SQLite)
DATABASE_URL = "sqlite:///./skin_platform.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

class UserDB(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True, index=True)
    email = Column(String, unique=True, index=True, nullable=False)
    password_hash = Column(String, nullable=False)

class ScanHistoryDB(Base):
    __tablename__ = "scan_history"
    id = Column(Integer, primary_key=True, index=True)
    user_email = Column(String, index=True, nullable=False)
    filename = Column(String)
    predicted_condition = Column(String)
    confidence = Column(Float)
    timestamp = Column(DateTime, default=datetime.utcnow)

Base.metadata.create_all(bind=engine)

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
security_bearer = HTTPBearer()
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

CLASS_NAMES = ['acne', 'blackheads', 'clearskin', 'darkspots', 'pores', 'wrinkles']
MODEL_PATH = os.path.join("models", "best_mobilenetv2_model.keras")
model = tf.keras.models.load_model(MODEL_PATH) if os.path.exists(MODEL_PATH) else None

# Initialize FastAPI App FIRST
app = FastAPI(title="DermAI Backend")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Rule Base Engine with Product Recommendations
RULE_BASE = {
    "acne": {
        "active_ingredients": ["Salicylic Acid (BHA 2%)", "Niacinamide 5%", "Zinc PCA"],
        "guideline": "Gentle chemical exfoliation to unclog pores without stripping skin moisture barrier.",
        "avoid": ["Heavy pore-clogging oils", "Physical facial scrubs", "Alcohol-based toners"],
        "recommended_products": [
            "CeraVe Acne Foaming Cream Cleanser",
            "The Ordinary Niacinamide 10% + Zinc 1%",
            "Paula's Choice 2% BHA Liquid Exfoliant"
        ]
    },
    "blackheads": {
        "active_ingredients": ["Salicylic Acid 2%", "Kaolin Clay Mask", "Niacinamide"],
        "guideline": "BHA penetrates oil-rich pores to dissolve blackhead plugs while clay absorbs excess sebum.",
        "avoid": ["Comedogenic creams", "Pore strips", "Heavy butter-based balms"],
        "recommended_products": [
            "COSRX BHA Blackhead Power Liquid",
            "Innisfree Super Volcanic Pore Clay Mask",
            "La Roche-Posay Effaclar Clarifying Solution"
        ]
    },
    "clearskin": {
        "active_ingredients": ["Hyaluronic Acid", "Vitamin C (L-Ascorbic Acid)", "Ceramides"],
        "guideline": "Maintain robust moisture barrier, antioxidant protection, and sun damage defense.",
        "avoid": ["Over-exfoliating", "Aggressive active stacking", "Skipping daily SPF"],
        "recommended_products": [
            "Cetaphil Gentle Skin Cleanser",
            "The Ordinary Hyaluronic Acid 2% + B5",
            "Beauty of Joseon Relief Sun SPF 50+"
        ]
    },
    "darkspots": {
        "active_ingredients": ["Vitamin C 15%", "Alpha Arbutin 2%", "Niacinamide", "Tranexamic Acid"],
        "guideline": "Target hyperpigmentation by suppressing melanin synthesis while enhancing cell turnover.",
        "avoid": ["Unprotected sun exposure", "Picking lesions", "Mixing harsh acids simultaneously"],
        "recommended_products": [
            "SkinCeuticals C E Ferulic or Minimalist Vitamin C 10%",
            "The Ordinary Alpha Arbutin 2% + HA",
            "Neutrogena Ultra Sheer Dry-Touch SPF 50+"
        ]
    },
    "pores": {
        "active_ingredients": ["Niacinamide 5-10%", "Peptides", "BHA 1%"],
        "guideline": "Regulate sebum flow, tighten enlarged pore appearance, and boost dermal elasticity.",
        "avoid": ["Heavy waxes", "Over-drying cleansers", "Silicones that trap debris"],
        "recommended_products": [
            "Paula's Choice 10% Niacinamide Booster",
            "SVR Sebiaclear Micro-Peel",
            "Bioderma Sebium Pore Refiner"
        ]
    },
    "wrinkles": {
        "active_ingredients": ["Encapsulated Retinol", "Multi-Peptide Complex", "Ceramides & Squalane"],
        "guideline": "Stimulate epidermal renewal and collagen density while deeply nourishing skin barrier.",
        "avoid": ["Dehydrating foaming washes", "Excessive UV exposure without SPF 50+", "Hot water washing"],
        "recommended_products": [
            "CeraVe Resurfacing Retinol Serum",
            "The Ordinary Multi-Peptide Serum",
            "CeraVe Moisturizing Cream with Ceramides"
        ]
    }
}

def get_db():
    db = SessionLocal()
    try: yield db
    finally: db.close()

async def verify_api_key(header_key: str = Security(api_key_header)):
    if header_key != API_KEY: raise HTTPException(status_code=401, detail="Invalid API Key")
    return header_key

async def get_current_user(credentials: HTTPAuthorizationCredentials = Depends(security_bearer)):
    try:
        payload = jwt.decode(credentials.credentials, JWT_SECRET_KEY, algorithms=[JWT_ALGORITHM])
        return payload.get("sub")
    except: raise HTTPException(status_code=401, detail="Invalid token")

class UserRegister(BaseModel):
    email: EmailStr
    password: str

@app.post("/api/v1/auth/register")
async def register(user: UserRegister, db: Session = Depends(get_db)):
    if db.query(UserDB).filter(UserDB.email == user.email).first():
        raise HTTPException(status_code=400, detail="User already exists")
    db.add(UserDB(email=user.email, password_hash=pwd_context.hash(user.password)))
    db.commit()
    return {"message": "User registered successfully"}

@app.post("/api/v1/auth/login")
async def login(user: UserRegister, db: Session = Depends(get_db)):
    db_user = db.query(UserDB).filter(UserDB.email == user.email).first()
    if not db_user or not pwd_context.verify(user.password, db_user.password_hash):
        raise HTTPException(status_code=401, detail="Invalid credentials")
    token = jwt.encode({"sub": db_user.email, "exp": datetime.utcnow() + timedelta(hours=24)}, JWT_SECRET_KEY, algorithm=JWT_ALGORITHM)
    return {"access_token": token}

@app.post("/api/v1/assess-skin/image")
async def analyze_image(file: UploadFile = File(...), user_email: str = Depends(get_current_user), db: Session = Depends(get_db)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB").resize((224, 224))
    img_arr = tf.keras.applications.mobilenet_v2.preprocess_input(np.expand_dims(tf.keras.preprocessing.image.img_to_array(image), axis=0))
    
    if model:
        preds = model.predict(img_arr)[0]
        top_idx = int(np.argmax(preds))
        condition, conf = CLASS_NAMES[top_idx], float(preds[top_idx])
    else: 
        condition, conf = "clearskin", 0.85

    db.add(ScanHistoryDB(user_email=user_email, filename=file.filename, predicted_condition=condition, confidence=round(conf, 4)))
    db.commit()

    return {"condition": condition, "confidence": round(conf, 4), "rules": RULE_BASE.get(condition, RULE_BASE["clearskin"])}

@app.post("/api/v1/assess-skin/7-day-plan")
async def get_7_day_plan(user_email: str = Depends(get_current_user), db: Session = Depends(get_db)):
    last_scan = db.query(ScanHistoryDB).filter(ScanHistoryDB.user_email == user_email).order_by(ScanHistoryDB.timestamp.desc()).first()
    if not last_scan:
        raise HTTPException(status_code=400, detail="No scan history found. Run an image scan first.")
    
    cond = last_scan.predicted_condition
    rules = RULE_BASE.get(cond, RULE_BASE["clearskin"])

    if gemini_client and GEMINI_API_KEY and GEMINI_API_KEY != "paste_your_copied_gemini_key_here":
        try:
            prompt = f"""
You are an expert clinical dermatologist. Create an exhaustive, highly detailed 7-Day Skincare Routine Plan for a patient diagnosed with '{cond.upper()}'.

Patient Diagnostics & Rules:
- Primary Actives: {', '.join(rules['active_ingredients'])}
- Clinical Focus: {rules['guideline']}
- Recommended Product Types & Examples: {', '.join(rules['recommended_products'])}
- Contraindications/Avoid: {', '.join(rules['avoid'])}

Produce a detailed schedule formatted as follows:

1. COMPREHENSIVE DERMATOLOGICAL ANALYSIS
   - Why these active ingredients were chosen for {cond}.
   - What physiological changes to expect over the 7 days.

2. RECOMMENDED OVER-THE-COUNTER PRODUCTS
   - Key product categories to buy (Cleansers, Serums, Moisturizers, SPF).
   - Recommended specific products: {', '.join(rules['recommended_products'])}.

3. DAY-BY-DAY DETAILED REGIMEN (Days 1 through 7)
   For EACH day, provide:
   - AM Routine (Step-by-step application order, exact waiting times in minutes between layers, and sunscreen recommendation).
   - PM Routine (Step-by-step cleansing, treatment actives application technique, and barrier recovery layer).
   - Daily Lifestyle & Hygiene Protocol (Pillowcases, water intake, diet triggers, towel care).

4. SAFETY & INGREDIENT CONTRAINDICATIONS
   - Detailed list of ingredients NEVER to mix with {rules['active_ingredients'][0]}.
   - How to identify and handle skin purging vs. an allergic reaction.
"""
            response = gemini_client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
            return {"condition": cond, "weekly_plan": response.text}
        except Exception:
            pass  # Fall back if API key fails or times out

    # Rule Fallback Plan
    fallback_plan = f"=== 7-DAY AI SKINCARE PLAN FOR: {cond.upper()} ===\n\n"
    fallback_plan += f"RECOMMENDED PRODUCTS:\n" + "\n".join([f"• {p}" for p in rules['recommended_products']]) + "\n\n"
    for day in range(1, 8):
        fallback_plan += f"DAY {day}:\n  • AM Routine: Gentle Cleanser -> {rules['active_ingredients'][0]} Serum -> Moisturizer -> Broad Spectrum SPF 50\n  • PM Routine: Double Cleanse -> {rules['active_ingredients'][-1]} -> Night Repair Cream\n\n"

    return {"condition": cond, "weekly_plan": fallback_plan}

Overwriting main.py


In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()
uvicorn.run("main:app", host="127.0.0.1", port=8000, reload=True)

INFO:     Will watch for changes in these directories: ['C:\\Users\\gurup\\OneDrive\\Desktop\\AI-Skin-Classifier']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [3116] using StatReload
